In [ ]:
# R1 -- inputs: each model's final scored file + test captions (source + lay reference). Peak RAM: ~60 MB per model
import torch   # must load before pandas/sacrebleu on Windows (avoids WinError 1114)
from pathlib import Path
import json, os, time, hashlib
import numpy as np
import pandas as pd

project_dir  = Path(r"D:\Project\Radiology_RAG")
TEST_DIR     = project_dir/"results"/"test"
TEST_PARQUET = project_dir/"data"/"MedLayBench-V"/"chest"/"test_chest_text_final.parquet"
MODELS = {"qwen2.5:1.5b": "qwen25_1.5b", "llama3.2:3b": "llama32_3b"}   # model -> file tag used by test_run.ipynb

md5 = lambda t: hashlib.md5(str(t).encode("utf-8")).hexdigest()
caps = pd.read_parquet(TEST_PARQUET, columns=["expert_caption", "lay_caption"])   # row number = sample_id

runs = {}
for model, tag in MODELS.items():
    f = TEST_DIR/f"test_kb_v2_{tag}_scored.csv"
    if not f.exists():
        print(f"skipped {model}: {f.name} not found yet")
        continue
    d = pd.read_csv(f, usecols=["model", "method", "sample_id", "answer"])
    d = d[d["model"] == model].reset_index(drop=True)
    assert len(d) == 16740 and d["sample_id"].nunique() == 2790, f"{model}: {len(d):,} rows, expected 16,740"
    d["answer"] = d["answer"].fillna("").astype(str)
    d["source"] = caps["expert_caption"].iloc[d["sample_id"]].to_numpy()   # original jargon caption
    d["lay"]    = caps["lay_caption"].iloc[d["sample_id"]].to_numpy()      # MedLayBench-V lay reference
    d["answer_md5"] = d["answer"].map(md5)                                 # ties every score to the exact answer text
    runs[model] = d
    print(f"{model}: {len(d):,} answers from {f.name}")
assert runs, "no scored file found in results/test"

In [ ]:
# R2 -- SARI, BLEU, ROUGE-L for every answer. CPU, ~1-3 min per model. Peak RAM: ~50 MB
# Reference = the lay caption (one reference per caption).
# SARI (Xu et al., 2016): same computation as the Hugging Face `evaluate` SARI metric -- lowercased, 13a tokens,
#   n-grams 1-4, keep F1 + delete precision + add F1, averaged. Mean of sentence SARI = evaluate's corpus SARI.
# BLEU (Papineni et al., 2002): sacrebleu sentence BLEU with default exp smoothing, used for the paired tests;
#   corpus BLEU per method is printed in R4 for comparison with other papers.
# ROUGE-L (Lin, 2004): F-measure with Porter stemming (rouge-score).
from collections import Counter
import sacrebleu
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
from rouge_score import rouge_scorer

tok13a = Tokenizer13a()

def ngrams(toks, n):
    return [" ".join(toks[i:i + n]) for i in range(len(toks) - n + 1)]

def sari_ngram(s, c, rlist, numref):
    rc = Counter(g for r in rlist for g in r)
    sc, cc = Counter(s), Counter(c)
    sc_rep = Counter({g: k * numref for g, k in sc.items()})
    cc_rep = Counter({g: k * numref for g, k in cc.items()})
    keep = sc_rep & cc_rep                       # kept from the source
    keep_good, keep_all = keep & rc, sc_rep & rc
    kp = sum(keep_good[g] / keep[g] for g in keep) / len(keep) if keep else 1
    kr = sum(keep_good[g] for g in keep) / sum(keep_all.values()) if keep_all else 1
    keep_f = 2 * kp * kr / (kp + kr) if (kp > 0 or kr > 0) else 0
    dele = sc_rep - cc_rep                       # deleted from the source
    del_good = dele - rc
    dp = sum(del_good[g] / dele[g] for g in dele) / len(dele) if dele else 1
    add = set(cc) - set(sc)                      # new words not in the source
    add_good, add_all = add & set(rc), set(rc) - set(sc)
    ap = len(add_good) / len(add) if add else 1
    ar = len(add_good) / len(add_all) if add_all else 1
    add_f = 2 * ap * ar / (ap + ar) if (ap > 0 or ar > 0) else 0
    return keep_f, dp, add_f

def sari_sent(src, pred, refs):
    s, c = tok13a(src.lower()).split(), tok13a(pred.lower()).split()
    rs = [tok13a(r.lower()).split() for r in refs]
    total = 0.0
    for n in (1, 2, 3, 4):
        total += sum(sari_ngram(ngrams(s, n), ngrams(c, n), [ngrams(r, n) for r in rs], len(rs)))
    return 100 * total / 12                      # mean of keep, delete, add over 4 n-gram orders

# self-check: worked example from the `evaluate` SARI metric card (expected 26.9536...)
_ex = sari_sent("About 95 species are currently accepted.", "About 95 you now get in.",
                ["About 95 species are currently known.", "About 95 species are now accepted.", "95 species are now accepted."])
assert abs(_ex - 26.953601953601954) < 1e-6, f"SARI check failed: {_ex}"

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
for model, d in runs.items():
    t0 = time.time()
    d["sari"]   = [sari_sent(s, a, [r]) for s, a, r in zip(d["source"], d["answer"], d["lay"])]
    d["bleu"]   = [sacrebleu.sentence_bleu(a, [r]).score for a, r in zip(d["answer"], d["lay"])]
    # score(target, prediction) returns {"rougeL": Score(precision, recall, fmeasure)}
    d["rougeL"] = [100 * rouge.score(r, a)["rougeL"].fmeasure for a, r in zip(d["answer"], d["lay"])]
    print(f"{model}: SARI / BLEU / ROUGE-L done in {time.time() - t0:.0f} s")
    display(d.groupby("method")[["sari", "bleu", "rougeL"]].mean().round(2))

In [ ]:
# R3 -- BERTScore (Zhang et al., 2020), answer vs lay reference. Saved chunk by chunk; a restart continues.
# Setting: roberta-large, layer 17 (package default for English), no idf, rescaled with the official baseline.
# First run downloads roberta-large (~1.4 GB). CPU time: roughly 1-2 h per model (estimate). Peak RAM: ~2 GB.
# Quit the Ollama app first: it is not needed here and frees memory.
from bert_score import BERTScorer

CHUNK = 500
torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))
scorer = BERTScorer(model_type="roberta-large", lang="en", rescale_with_baseline=True, batch_size=32)
print("BERTScore setting:", scorer.hash)        # write this string in BAB III

for model, d in runs.items():
    cache = TEST_DIR/f"test_kb_v2_{MODELS[model]}_bertscore.jsonl"
    saved = {}
    if cache.exists():
        raw = cache.read_bytes()
        *lines, tail = raw.split(b"\n")
        if tail.strip():                         # last line only half-written when the run stopped
            with open(cache, "r+b") as f:
                f.truncate(len(raw) - len(tail))
            print(f"repaired {cache.name}: removed 1 half-written line")
        for line in lines:
            if line.strip():
                r = json.loads(line)
                saved[(r["method"], r["sample_id"], r["answer_md5"])] = r
    keys = list(zip(d["method"], d["sample_id"].astype(int), d["answer_md5"]))
    todo = [i for i, k in enumerate(keys) if k not in saved]
    print(f"{model}: BERTScore already saved {len(d) - len(todo):,}/{len(d):,} | to score now: {len(todo):,}")

    t0 = time.time()
    with open(cache, "a", encoding="utf-8", newline="\n") as out:
        for s in range(0, len(todo), CHUNK):
            idx = todo[s:s + CHUNK]
            P, R, F = scorer.score(d["answer"].iloc[idx].tolist(), d["lay"].iloc[idx].tolist())
            for j, i in enumerate(idx):
                rec = {"method": keys[i][0], "sample_id": keys[i][1], "answer_md5": keys[i][2],
                       "bertscore_p": P[j].item(), "bertscore_r": R[j].item(), "bertscore_f1": F[j].item()}
                out.write(json.dumps(rec) + "\n")
                saved[keys[i]] = rec
            out.flush()
            os.fsync(out.fileno())                 # this chunk is on disk before the next one starts
            done, el = s + len(idx), time.time() - t0
            print(f"{time.strftime('%H:%M')} | {model} | {done:,}/{len(todo):,} | ~{(len(todo) - done) * el / done / 3600:.1f} h left")

    for col in ["bertscore_p", "bertscore_r", "bertscore_f1"]:
        d[col] = [saved[k][col] for k in keys]
    display(d.groupby("method")[["bertscore_f1"]].mean().round(3))

In [ ]:
# R4 -- one reference-metrics file per model + corpus BLEU per method. Peak RAM: ~0 MB
COLS = ["model", "method", "sample_id", "answer_md5", "sari", "bleu", "rougeL",
        "bertscore_p", "bertscore_r", "bertscore_f1"]
for model, d in runs.items():
    out = TEST_DIR/f"test_kb_v2_{MODELS[model]}_refmetrics.csv"
    d[[c for c in COLS if c in d]].to_csv(out, index=False)
    summ = d.groupby("method")[[c for c in ["sari", "bleu", "rougeL", "bertscore_f1"] if c in d]].mean().round(3)
    summ["corpus_bleu"] = pd.Series({m: sacrebleu.corpus_bleu(g["answer"].tolist(), [g["lay"].tolist()]).score
                                     for m, g in d.groupby("method")}).round(2)
    print(f"{model} -> {out}")
    display(summ)